## 03-transformer-block: Experiment 노트북

### 목표
- Scaled Dot-Product Attention에서 $\sqrt{d_k}$의 필요성 검증

### 1. Import 및 설정

In [2]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
import re
import math
import copy
import random
from functools import partial
from typing import Tuple, Dict, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import TensorDataset, DataLoader
from datasets import load_dataset

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from attention import scaled_dot_product_attention

### 2. Dot-Product Attention에서 $\sqrt{d_k}$ 스케일링의 효과 확인

In [7]:
torch.manual_seed(42)

batch_size = 64
seq_len = 32
dim_k_values = [8, 32, 128]

for dim_k in dim_k_values:
    query = torch.randn(batch_size, seq_len, dim_k)
    key = torch.randn(batch_size, seq_len, dim_k)

    scores = query @ key.transpose(-1, -2)
    scaled_scores = scores / math.sqrt(dim_k)

    score_std = scores.std(unbiased=False).item()
    scaled_score_std = scaled_scores.std(unbiased=False).item()

    print(
        f"D_k={dim_k:>3} | "
        f"before={score_std:.4f} | "
        f"after={scaled_score_std:.4f}"
    )

    assert abs(scaled_score_std - 1.0) < 0.1

    unscaled_weights = scores.softmax(dim=-1)
    scaled_weights = scaled_scores.softmax(dim=-1)

    # 각 query가 가장 크게 선택한 확률의 평균
    unscaled_max_prob = unscaled_weights.max(dim=-1).values.mean()
    scaled_max_prob = scaled_weights.max(dim=-1).values.mean()

    # 분포의 평균 entropy
    # 평균 entropy = 각 key의 정보량(-log p)을 선택 확률(p)로 가중 평균한 값
    unscaled_entropy = -(
        unscaled_weights
        * unscaled_weights.clamp_min(1e-12).log()
    ).sum(dim=-1).mean()

    scaled_entropy = -(
        scaled_weights
        * scaled_weights.clamp_min(1e-12).log()
    ).sum(dim=-1).mean()

    print(
        f"D_k={dim_k:>3} | "
        f"max prob: {unscaled_max_prob:.4f} → {scaled_max_prob:.4f} | "
        f"entropy: {unscaled_entropy:.4f} → {scaled_entropy:.4f}"
    )

D_k=  8 | before=2.8439 | after=1.0055
D_k=  8 | max prob: 0.4977 → 0.1638 | entropy: 1.6470 → 3.0174
D_k= 32 | before=5.6850 | after=1.0050
D_k= 32 | max prob: 0.7429 → 0.1650 | entropy: 0.7359 → 3.0112
D_k=128 | before=11.3239 | after=1.0009
D_k=128 | max prob: 0.8677 → 0.1655 | entropy: 0.3434 → 3.0099


#### 결과

- `D_k`가 커질수록 scaling을 적용하지 않은 attention score의 표준편차가 증가
  - 그 결과 softmax의 최대 확률은 증가하고 entropy는 감소하여 분포가 한 key에 포화된 것을 확인
  - softmax의 최대 확률이 증가한다는 것은, 확률분포가 더욱 뾰족해지는 것으로 볼 수 있므며, 따라서 모든 토큰을 골고루 보지 않고 one-hot과 같은 형태로 토큰을 보게 되어 attention의 가치가 떨어진다.
- 반면 `1 / sqrt(D_k)` scaling을 적용하면 score의 표준편차가 약 1로 유지됐으며, 최대 확률과 entropy도 `D_k`와 관계없이 안정적으로 유지됨